# 32 — Regression Models (Merged Feature Set)

Retrain regression models using the same feature pipeline as the deployment service (`src/`).  
This replaces notebook 31 which used an older, pre-saved feature set that lacked Scopus journal metrics.

**Key changes vs notebook 31:**
- Loads `all_unis_cleaned.pkl` (merged institution data)
- Uses `CitationPredictor.prepare_features()` directly — guarantees training/inference feature parity
- Includes Scopus metrics: CiteScore, SJR, SNIP and their percentiles
- Better LightGBM hyperparameters
- Saves `citation_threshold_p75.pkl` so the app derives `is_high_impact` from regression output

In [ ]:
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')

# Paths
PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH      = PROJECT_ROOT / 'data' / 'processed' / 'all_unis_cleaned.pkl'
FEATURES_DIR   = PROJECT_ROOT / 'data' / 'features'
MODELS_DIR     = PROJECT_ROOT / 'models' / 'regression'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE   = 42
TRAIN_YEARS    = list(range(2015, 2018))   # 2015–2017
TEST_YEARS     = list(range(2018, 2021))   # 2018–2020
TFIDF_MAX_FEAT = 5000
print('Paths OK')

## 1. Load & split data

In [ ]:
df = pd.read_pickle(DATA_PATH)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
print(df.dtypes[['Year','Citations']].to_string() if 'Citations' in df.columns else 'No Citations column!')
df.head(2)

In [ ]:
# Temporal split — restrict to AUB papers (consistent with earlier notebooks)
aub_mask = df['Institution'].str.upper().str.contains('BEIRUT|AUB', na=False) \
           if 'Institution' in df.columns else pd.Series(True, index=df.index)

df_aub = df[aub_mask].copy()

df_train = df_aub[df_aub['Year'].isin(TRAIN_YEARS)].copy()
df_test  = df_aub[df_aub['Year'].isin(TEST_YEARS)].copy()

# Drop rows without abstract or citations
df_train = df_train.dropna(subset=['Abstract', 'Citations'])
df_test  = df_test.dropna(subset=['Abstract', 'Citations'])

print(f'Train: {len(df_train):,} papers ({TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]})')
print(f'Test:  {len(df_test):,} papers ({TEST_YEARS[0]}–{TEST_YEARS[-1]})')
print(f'Train citations — median: {df_train["Citations"].median():.0f}, '
      f'mean: {df_train["Citations"].mean():.1f}, '
      f'p75: {df_train["Citations"].quantile(0.75):.0f}')

## 2. Fit TF-IDF & save artifacts

In [ ]:
# Fit TF-IDF on training abstracts only, then save
tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEAT,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,          # strip near-universal terms
    stop_words='english', # remove stop words so 'the','in','of' don't dominate
    sublinear_tf=True
)
tfidf.fit(df_train_p['Abstract'].fillna(''))

with open(FEATURES_DIR / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print(f'TF-IDF fitted ({tfidf.max_features} features) and saved.')

In [ ]:
# Compute & save venue statistics from training data
from src.features.venue_features import compute_venue_statistics

venue_stats = compute_venue_statistics(df_train, venue_col='Scopus Source title', citation_col='Citations')
with open(FEATURES_DIR / 'venue_statistics.pkl', 'wb') as f:
    pickle.dump(venue_stats, f)
print(f'Venue statistics saved ({len(venue_stats["paper_counts"])} venues).')

## 3. Build features using the deployment pipeline

In [ ]:
from src.deployment.prediction_service import CitationPredictor

predictor = CitationPredictor(
    models_dir=str(PROJECT_ROOT / 'models'),
    features_dir=str(FEATURES_DIR)
)
predictor.initialize()
print('Predictor initialised.')

In [ ]:
# Rename columns to match what prepare_features() expects
COL_MAP = {
    'Source title': 'Scopus Source title',
    'H-index':      'Authors H-index',
    'Publication type': 'Publication Type',
    'Source type':  'Source type',
    'Open Access':  'is_open_access',
    'Topic Prominence Percentile': 'topic_prominence',
}

def prep_df(df_in):
    d = df_in.copy().reset_index(drop=True)  # contiguous RangeIndex to match X output
    for old, new in COL_MAP.items():
        if old in d.columns and new not in d.columns:
            d.rename(columns={old: new}, inplace=True)
    if 'is_open_access' in d.columns:
        d['is_open_access'] = pd.to_numeric(d['is_open_access'].map(
            lambda x: 1 if str(x).strip().lower() not in ('', 'nan', 'none', 'no', '0', 'false') else 0
        ), errors='coerce').fillna(0).astype(int)
    if 'topic_prominence' not in d.columns and 'Topic Prominence Percentile' in df_in.columns:
        d['topic_prominence'] = pd.to_numeric(df_in['Topic Prominence Percentile'].values, errors='coerce')
        d['topic_prominence'] = d['topic_prominence'].fillna(50.0)
    return d

df_train_p = prep_df(df_train)
df_test_p  = prep_df(df_test)

# Keep aligned citation arrays (same reset index as df_train_p)
citations_train = df_train['Citations'].reset_index(drop=True)
citations_test  = df_test['Citations'].reset_index(drop=True)
print('Column mapping done.')

In [ ]:
X_train = predictor.prepare_features(df_train_p)
X_test  = predictor.prepare_features(df_test_p)

# Drop features that leak citation-label information into training:
#   venue_avg_citations  — computed from training citations (direct target encoding)
#   venue_prestige_score — derived from avg_citations
#   is_top_venue         — also derived from avg_citations
# These cause ~0.5 train/test Spearman gap and near-zero High-Impact recall.
# At inference the saved model simply won't expect these columns, so they are ignored.
LEAKY = ['venue_avg_citations', 'venue_prestige_score', 'is_top_venue']
dropped = [c for c in LEAKY if c in X_train.columns]
X_train = X_train.drop(columns=dropped)
X_test  = X_test.drop(columns=dropped)
print(f'Dropped leaky venue columns: {dropped}')
print(f'Kept: venue_paper_count = {("venue_paper_count" in X_train.columns)}')

# Align targets to feature index — prepare_features may drop rows
y_train = np.log1p(citations_train.loc[X_train.index].values)
y_test  = np.log1p(citations_test.loc[X_test.index].values)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Rows dropped — train: {len(df_train_p) - len(X_train)}, test: {len(df_test_p) - len(X_test)}')
print(f'y_train range: [{y_train.min():.2f}, {y_train.max():.2f}]')

In [ ]:
X_train = predictor.prepare_features(df_train_p)
X_test  = predictor.prepare_features(df_test_p)

# Align targets to feature index — prepare_features may drop rows (e.g. all-NaN abstracts)
y_train = np.log1p(citations_train.loc[X_train.index].values)
y_test  = np.log1p(citations_test.loc[X_test.index].values)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Rows dropped — train: {len(df_train_p) - len(X_train)}, test: {len(df_test_p) - len(X_test)}')
print(f'y_train range: [{y_train.min():.2f}, {y_train.max():.2f}]')

## 4. Train LightGBM regressor

In [ ]:
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[__import__('lightgbm').early_stopping(50, verbose=False),
               __import__('lightgbm').log_evaluation(100)]
)
print(f'Best iteration: {model.best_iteration_}')

## 5. Evaluate

In [ ]:
def evaluate(y_true_log, y_pred_log, label=''):
    rmse = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    mae  = mean_absolute_error(y_true_log, y_pred_log)
    r2   = r2_score(y_true_log, y_pred_log)
    rho  = spearmanr(y_true_log, y_pred_log).statistic

    # Also evaluate on original scale
    y_true_raw = np.expm1(y_true_log)
    y_pred_raw = np.expm1(y_pred_log)
    mae_raw    = mean_absolute_error(y_true_raw, y_pred_raw)
    r2_raw     = r2_score(y_true_raw, y_pred_raw)

    print(f'{label}')
    print(f'  Log-space  — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R²: {r2:.4f}  Spearman: {rho:.4f}')
    print(f'  Raw-space  — MAE: {mae_raw:.1f} citations  R²: {r2_raw:.4f}')
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'spearman': rho, 'mae_raw': mae_raw, 'r2_raw': r2_raw}

train_metrics = evaluate(y_train, model.predict(X_train), 'Train')
test_metrics  = evaluate(y_test,  model.predict(X_test),  'Test')

In [ ]:
# Regression-derived classification accuracy at the p75 threshold
thr_75 = citations_train.loc[X_train.index].quantile(0.75)
print(f'p75 citation threshold (train): {thr_75:.0f}')

y_test_raw  = np.expm1(y_test)
y_pred_raw  = np.expm1(model.predict(X_test))

true_label  = (y_test_raw  >= thr_75).astype(int)
pred_label  = (y_pred_raw  >= thr_75).astype(int)

from sklearn.metrics import classification_report
print(classification_report(true_label, pred_label, target_names=['Not High Impact', 'High Impact']))

## 6. Save model & threshold

In [ ]:
with open(MODELS_DIR / 'lightgbm.pkl', 'wb') as f:
    pickle.dump(model, f)
print(f'Model saved to {MODELS_DIR / "lightgbm.pkl"}')

with open(MODELS_DIR / 'citation_threshold_p75.pkl', 'wb') as f:
    pickle.dump(float(thr_75), f)
print(f'Threshold saved: {thr_75:.0f} citations (p75 of training set)')

In [ ]:
# Top-20 most important features
import pandas as pd
imp = pd.Series(model.feature_importances_, index=X_train.columns)
imp.nlargest(20).sort_values().plot(kind='barh', figsize=(8, 6), title='Top 20 Feature Importances')
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

## Summary

| Artifact | Path |
|---|---|
| Regression model | `models/regression/lightgbm.pkl` |
| p75 threshold | `models/regression/citation_threshold_p75.pkl` |
| TF-IDF vectorizer | `data/features/tfidf_vectorizer.pkl` |
| Venue statistics | `data/features/venue_statistics.pkl` |

The app will automatically pick up `citation_threshold_p75.pkl` and switch from the standalone
classifier to regression-derived `is_high_impact`.